In [53]:

from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

2- Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

5- Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

6- Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [23]:
import nltk
from nltk.corpus import gutenberg
import os

nltk.download('gutenberg')

data = gutenberg.raw('shakespeare-hamlet.txt')

file_path = '/content/drive/MyDrive/lstm implementation/hamlet.txt'

os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(data)

print("Hamlet file saved successfully!")

Hamlet file saved successfully!


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [24]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [25]:
with open('/content/drive/MyDrive/lstm implementation/hamlet.txt', 'r', encoding='utf-8') as file:
  text=file.read().lower()

tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
total_words


4818

In [26]:
#creating input sequence
inputseq=[]
for line in text.split('\n'):
  token_list=tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    n_gram_sequence=token_list[:i+1]
    inputseq.append(n_gram_sequence)

In [27]:
#pad req
max_sequence_len=max([len(x) for x in inputseq])
inputseq=np.array(pad_sequences(inputseq,maxlen=max_sequence_len,padding='pre'))

In [28]:
inputseq

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [29]:
#creating predictors and label
import tensorflow as tf
x,y=inputseq[:,:-1],inputseq[:,-1]

In [30]:
y=tf.keras.utils.to_categorical(y,num_classes=total_words)

In [31]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [37]:
#Train lstm rnn
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

#Define the model
model=Sequential()
model.add(Embedding(total_words,100,input_shape=(max_sequence_len - 1,)))
model.add(LSTM(150))
model.add(Dropout(0.2))
model.add(Dense(total_words,activation='softmax'))

#compile the model
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()



Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4818)           │       727,518 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,359,918 (5.19 MB)

 Trainable params: 1,359,918 (5.19 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [34]:
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

print("Training target distribution:")
print(y_train.sum(axis=0).max(), y_train.sum(axis=0).min())

x_train: (20585, 13)
y_train: (20585, 4818)
x_test: (5147, 13)
y_test: (5147, 4818)
Training target distribution:
709.0 0.0


In [40]:
#train the model
history=model.fit(x_train,y_train,epochs=100,batch_size=128,validation_data=(x_test,y_test),verbose=1)

Epoch 1/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.0361 - loss: 6.5008 - val_accuracy: 0.0404 - val_loss: 6.7854
Epoch 2/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0414 - loss: 6.3993 - val_accuracy: 0.0416 - val_loss: 6.8459
Epoch 3/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0451 - loss: 6.2995 - val_accuracy: 0.0472 - val_loss: 6.8808
Epoch 4/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0505 - loss: 6.2108 - val_accuracy: 0.0499 - val_loss: 6.8832
Epoch 5/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0535 - loss: 6.1179 - val_accuracy: 0.0525 - val_loss: 6.9003
Epoch 6/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0569 - loss: 6.0369 - val_accuracy: 0.0546 - val_loss: 6.9201
Epoch 7/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0577 - loss: 5.9592 - val_accuracy: 0.0546 - val_loss: 6.9436
Epoch 8/100
161/161 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0614 - loss: 5.8806 - 

In [45]:
#Predicting the next word
def predict_next_word(model,tokenizer,text,max_sequence_len):
  token_list=tokenizer.texts_to_sequences([text])[0]
  if len(token_list)>=max_sequence_len:
    token_list=token_list[-(max_sequence_len-1):]
  token_list=pad_sequences([token_list],maxlen=max_sequence_len-1,padding='pre')
  predicted=model.predict(token_list,verbose=0)
  predicted_word_index=np.argmax(predicted,axis=1)
  for word,index in tokenizer.word_index.items():
    if index==predicted_word_index:
      return word

In [48]:
input_text="to be or not to"
print(f"input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"Next word predicition:{next_word}")

input text:to be or not to
Next word predicition:his


In [54]:
import os
import pickle

# Your Google Drive folder
save_path = "/content/drive/MyDrive/lstm implementation"

# Save the trained model
model.save(os.path.join(save_path, "next_word_prediction.keras"))

# Save the tokenizer
with open(os.path.join(save_path, "tokenizer.pickle"), "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!
